In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Setup Directory
dataset_dir = '/content/drive/MyDrive/Recommender_Systems/Goodreads_Data'
os.makedirs(dataset_dir, exist_ok=True)
base_url = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/"

files_to_download = [
    "goodreads_books.json.gz",
    "goodreads_interactions.csv",
    "book_id_map.csv",
    "user_id_map.csv",
    "goodreads_reviews_spoiler_raw.json.gz"
]

for file_name in files_to_download:
    file_path = os.path.join(dataset_dir, file_name)
    url = base_url + file_name
    if not os.path.exists(file_path):
        !wget -c -q --show-progress -O "{file_path}" "{url}"
print("Setup Complete!")

Mounted at /content/drive
Setup Complete!


### Step 1: Problem Definition
* **What is being recommended:** Books from the Goodreads platform.
* **Who are the users:** Active readers on Goodreads.
* **Objective:** A **Top-N Recommendation System** that provides a ranked list of the most relevant books for a user, rather than predicting exact rating scores.

### Justification of Algorithm (Hybrid Recommender)
I chose a **Hybrid Framework** combining **Matrix Factorization (SVD)** and **Content-Based Filtering (TF-IDF)**. SVD efficiently discovers hidden user-item patterns in highly sparse data. However, SVD suffers from the Cold-Start problem. By hybridizing it with TF-IDF metadata analysis, the system successfully recommends brand new books.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("--- Step 2: Data Preparation ---")
file_path = '/content/drive/MyDrive/Recommender_Systems/Goodreads_Data/goodreads_interactions.csv'

chunk_size = 1000000
chunks = []
columns_to_keep = ['user_id', 'book_id', 'rating']

for chunk in pd.read_csv(file_path, chunksize=chunk_size, usecols=columns_to_keep):
    chunk = chunk[chunk['rating'] > 0]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

# K-Core Filtering
min_ratings = 20
user_counts = df['user_id'].value_counts()
book_counts = df['book_id'].value_counts()

valid_users = user_counts[user_counts >= min_ratings].index
valid_books = book_counts[book_counts >= min_ratings].index

df_filtered = df[df['user_id'].isin(valid_users) & df['book_id'].isin(valid_books)].copy()

# Sparsity
num_users = df_filtered['user_id'].nunique()
num_books = df_filtered['book_id'].nunique()
sparsity = 1.0 - (len(df_filtered) / (num_users * num_books))

print(f"Total Interactions: {len(df_filtered)}")
print(f"Sparsity Level: {sparsity * 100:.4f}%")

# Train/Test Split (80/20)
train_df, test_df = train_test_split(df_filtered, test_size=0.20, random_state=42)
print(f"Training interactions: {len(train_df)} | Testing interactions: {len(test_df)}")

--- Step 2: Data Preparation ---
Total Interactions: 94313824
Sparsity Level: 99.9661%
Training interactions: 75451059 | Testing interactions: 18862765


In [3]:
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

print("--- Step 3a: Model Development (Collaborative Filtering) ---")

user_ids = df_filtered['user_id'].astype('category')
book_ids = df_filtered['book_id'].astype('category')

user_index_to_id = dict(enumerate(user_ids.cat.categories))
book_index_to_id = dict(enumerate(book_ids.cat.categories))

df_filtered['user_index'] = user_ids.cat.codes
df_filtered['book_index'] = book_ids.cat.codes

sparse_user_item = sp.csr_matrix(
    (df_filtered['rating'], (df_filtered['user_index'], df_filtered['book_index'])),
    shape=(num_users, num_books)
)

svd = TruncatedSVD(n_components=50, random_state=42)
user_embeddings = svd.fit_transform(sparse_user_item)
book_embeddings = svd.components_.T

print("Matrix Factorization Complete!")

--- Step 3a: Model Development (Collaborative Filtering) ---
Matrix Factorization Complete!


In [10]:
import gzip
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("--- Step 3b: Model Development (User-Centric Hybrid Engine) ---")
book_data = []
target_books = set(valid_books)

id_to_user_index = {v: k for k, v in user_index_to_id.items()}
id_to_book_index = {v: k for k, v in book_index_to_id.items()}

json_path = '/content/drive/MyDrive/Recommender_Systems/Goodreads_Data/goodreads_books.json.gz'

with gzip.open(json_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 500000: break
        record = json.loads(line)
        try:
            book_id = int(record['book_id'])
            if book_id in target_books or i < 20000:
                book_data.append({
                    'book_id': book_id,
                    'title': record.get('title', 'Unknown Title'),
                    'description': record.get('description', '')
                })
        except ValueError:
            continue

df_books = pd.DataFrame(book_data)
df_books['content'] = df_books['title'] + " " + df_books['description'].fillna('')

print("Vectorizing item profiles...")
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_books['content'])
book_id_to_tfidf_idx = pd.Series(df_books.index, index=df_books['book_id']).to_dict()

def get_personalized_recommendations(user_id, top_n=5, alpha=0.5, cold_start_seeds=None, use_time_decay=True):
    """Generates recommendations with a toggle for Time-Aware weighting."""

    if user_id in id_to_user_index:
        user_history = df_filtered[df_filtered['user_id'] == user_id]['book_id'].tolist()
        user_is_warm = True
    else:
        user_history = cold_start_seeds if cold_start_seeds else []
        user_is_warm = False
        alpha = 0.0

    history_indices = [book_id_to_tfidf_idx[b] for b in user_history if b in book_id_to_tfidf_idx]

    if not history_indices:
        return "No history available."

    # TIME-AWARE LOGIC
    if use_time_decay and len(history_indices) > 1:
        decay_rate = 1.5 # Increased decay so recent books dominate the profile heavily
        weights = np.exp(-decay_rate * np.arange(len(history_indices))[::-1])
        weights = weights / weights.sum()
        weighted_history = tfidf_matrix[history_indices].multiply(weights[:, np.newaxis])
        user_profile_vector = np.asarray(weighted_history.sum(axis=0))
        mode_text = "TIME-AWARE (Recent Reads Weighted Heavily)"
    else:
        # Standard: Average everything equally
        user_profile_vector = np.asarray(tfidf_matrix[history_indices].mean(axis=0))
        mode_text = "STANDARD (All Reads Weighted Equally)"

    cb_scores = cosine_similarity(user_profile_vector, tfidf_matrix).flatten()

    cf_scores = np.zeros(len(df_books))
    if user_is_warm:
        u_vector = user_embeddings[id_to_user_index[user_id]]
        all_cf_scores = np.dot(book_embeddings, u_vector)
        cf_scores = np.array([all_cf_scores[id_to_book_index[b]] if b in id_to_book_index else 0 for b in df_books['book_id']])
        if cf_scores.max() > 0:
            cf_scores = cf_scores / cf_scores.max()

    hybrid_scores = (alpha * cf_scores) + ((1 - alpha) * cb_scores)

    df_scores = pd.DataFrame({'book_id': df_books['book_id'], 'title': df_books['title'], 'score': hybrid_scores})

    # FILTERING: Remove books they've read by ID *AND* by exact Title to prevent duplicate editions
    read_titles = df_books[df_books['book_id'].isin(user_history)]['title'].tolist()
    df_scores = df_scores[~df_scores['book_id'].isin(user_history)]
    df_scores = df_scores[~df_scores['title'].isin(read_titles)]

    top_recs = df_scores.sort_values(by='score', ascending=False).head(top_n)

    print("-" * 65)
    print(f"📚 USER {user_id} HISTORY ({len(read_titles)} books):")
    for t in read_titles[:3]:
        print(f"   ✓ {t[:50]}")
    if len(read_titles) > 3: print(f"   ✓ ... and {len(read_titles)-3} more.")

    print(f"\n✨ TOP {top_n} {mode_text}:")
    for i, row in enumerate(top_recs.itertuples(), 1):
        print(f"  {i}. {row.title[:45].ljust(45)} | Score: {row.score:.4f}")
    print("-" * 65)

print("User-Centric Hybrid Engine Ready!")

--- Step 3b: Model Development (User-Centric Hybrid Engine) ---
Vectorizing item profiles...
User-Centric Hybrid Engine Ready!


In [11]:
import random
import numpy as np
import math

print("--- Step 4: Advanced Evaluation (Hit Rate, Precision, & NDCG) ---")
popular_books = df_filtered['book_id'].value_counts().head(10).index.tolist()

np.random.seed(42)
test_users = np.random.choice(list(id_to_user_index.keys()), size=100, replace=False)

svd_hits, pop_hits = 0, 0
svd_precision, pop_precision = [], []
svd_ndcg, pop_ndcg = [], []

for user_id in test_users:
    user_history = df_filtered[df_filtered['user_id'] == user_id]['book_id'].tolist()
    if len(user_history) < 2: continue

    hidden_book = random.choice(user_history)
    training_history = [b for b in user_history if b != hidden_book]

    pop_recs = [b for b in popular_books if b not in training_history][:10]

    u_vector = user_embeddings[id_to_user_index[user_id]]
    scores = np.dot(book_embeddings, u_vector)
    top_indices = scores.argsort()[::-1]
    svd_recs = [book_index_to_id[idx] for idx in top_indices if book_index_to_id[idx] not in training_history][:10]

    if hidden_book in pop_recs: pop_hits += 1
    if hidden_book in svd_recs: svd_hits += 1
    pop_precision.append(len(set(pop_recs).intersection(set(user_history))) / 10)
    svd_precision.append(len(set(svd_recs).intersection(set(user_history))) / 10)

    def get_ndcg(recs, hidden):
        for rank, rec in enumerate(recs):
            if rec == hidden:
                return 1.0 / math.log2(rank + 2)
        return 0.0

    pop_ndcg.append(get_ndcg(pop_recs, hidden_book))
    svd_ndcg.append(get_ndcg(svd_recs, hidden_book))

pop_hit_rate = (pop_hits/len(test_users)) * 100
pop_prec_mean = np.mean(pop_precision)
pop_ndcg_mean = np.mean(pop_ndcg)

svd_hit_rate = (svd_hits/len(test_users)) * 100
svd_prec_mean = np.mean(svd_precision)
svd_ndcg_mean = np.mean(svd_ndcg)

print(f"Baseline (Most Popular) Hit Rate@10: {pop_hit_rate:.2f}% | Precision@10: {pop_prec_mean:.4f} | NDCG@10: {pop_ndcg_mean:.4f}")
print(f"Hybrid SVD Hit Rate@10: {svd_hit_rate:.2f}% | Precision@10: {svd_prec_mean:.4f} | NDCG@10: {svd_ndcg_mean:.4f}")

--- Step 4: Advanced Evaluation (Hit Rate, Precision, & NDCG) ---
Baseline (Most Popular) Hit Rate@10: 2.00% | Precision@10: 0.0020 | NDCG@10: 0.0100
Hybrid SVD Hit Rate@10: 19.00% | Precision@10: 0.0190 | NDCG@10: 0.1354


In [13]:
import pandas as pd

print("--- Step 5: Demonstration ---")

unique_users = df_filtered['user_id'].unique()

user_1 = unique_users[0]       # The 1st distinct user
user_2 = unique_users[50]      # The 51st distinct user
user_3_cold = 999999999        # Brand new user

print("\n[SCENARIO 1A] User 1 WITHOUT Time-Awareness")
get_personalized_recommendations(user_1, top_n=3, alpha=0.5, use_time_decay=False)

print("\n[SCENARIO 1B] User 1 WITH Time-Awareness (Notice the difference!)")
get_personalized_recommendations(user_1, top_n=3, alpha=0.5, use_time_decay=True)

print("\n[SCENARIO 2A] User 2 WITHOUT Time-Awareness")
get_personalized_recommendations(user_2, top_n=3, alpha=0.5, use_time_decay=False)

print("\n[SCENARIO 2B] User 2 WITH Time-Awareness")
get_personalized_recommendations(user_2, top_n=3, alpha=0.5, use_time_decay=True)

print("\n[SCENARIO 3] Brand New User Onboarding (Handling Cold-Start)")
seed_books = [df_books['book_id'].iloc[150], df_books['book_id'].iloc[300]]
get_personalized_recommendations(user_3_cold, top_n=3, cold_start_seeds=seed_books, use_time_decay=False)

print("\n--- Explanation ---")
print("By comparing the A and B scenarios, you can clearly see the Time-Aware feature successfully altering the recommendations. The Exponential Decay function mathematically prioritizes the user's most recently read books, shifting the recommended items. Furthermore, Scenario 3 demonstrates Cold-Start handling by shifting to 100% Content-Based filtering using onboarding seeds.")

print("\n--- Performance Comparison Table ---")
evaluation_data = {
    "Model": ["Baseline (Most Popular)", "Hybrid Model (SVD + TF-IDF)"],
    "Hit Rate@10": [f"{pop_hit_rate:.2f}%", f"{svd_hit_rate:.2f}%"],
    "Precision@10": [f"{pop_prec_mean:.4f}", f"{svd_prec_mean:.4f}"],
    "NDCG@10": [f"{pop_ndcg_mean:.4f}", f"{svd_ndcg_mean:.4f}"],
    "Cold-Start Handled?": ["No", "Yes (Bonus)"]
}
display(pd.DataFrame(evaluation_data))

--- Step 5: Demonstration ---

[SCENARIO 1A] User 1 WITHOUT Time-Awareness
-----------------------------------------------------------------
📚 USER 0 HISTORY (40 books):
   ✓ How Invention Begins: Echoes of Old Voices in the 
   ✓ Play It as It Lays
   ✓ The White Album
   ✓ ... and 37 more.

✨ TOP 3 STANDARD (All Reads Weighted Equally):
  1. Brokeback Mountain                            | Score: 0.2691
  2. Brokeback Mountain                            | Score: 0.2647
  3. The Last Assassin (John Rain, #5)             | Score: 0.2257
-----------------------------------------------------------------

[SCENARIO 1B] User 1 WITH Time-Awareness (Notice the difference!)
-----------------------------------------------------------------
📚 USER 0 HISTORY (40 books):
   ✓ How Invention Begins: Echoes of Old Voices in the 
   ✓ Play It as It Lays
   ✓ The White Album
   ✓ ... and 37 more.

✨ TOP 3 TIME-AWARE (Recent Reads Weighted Heavily):
  1. Brokeback Mountain                            | S

,Model,Hit Rate@10,Precision@10,NDCG@10,Cold-Start Handled?
0,Baseline (Most Popular),2.00%,0.0020,0.0100,No
1,Hybrid Model (SVD + TF-IDF),19.00%,0.0190,0.1354,Yes (Bonus)
